好的！以下是《动手学深度学习》第11.10节 **Adam优化算法** 的完整学习笔记，已整理为 **Markdown 格式**，内容包括原理、公式、代码与对比，便于你复制到 Jupyter Notebook 或 Word 文档中。

---

# 📘 11.10 Adam算法 - 学习笔记

## 📌 一、Adam 是什么？

**Adam（Adaptive Moment Estimation）** 是一种结合了：

* **动量法（Momentum）**
* **RMSProp**

的优化算法。

> 它既使用梯度的\*\*一阶矩（均值）**来加速下降，又使用**二阶矩（方差）\*\*自适应调整学习率。

---

## ⚙️ 二、核心思想与公式

Adam 同时维护了两个滑动平均：

* \$\boldsymbol{v}\_t\$：梯度的一阶动量（类似 Momentum）
* \$\boldsymbol{s}\_t\$：梯度的二阶动量（类似 RMSProp）

### 🔢 更新步骤如下：

1. **一阶动量（梯度的指数加权平均）：**

   $$
   \boldsymbol{v}_t = \beta_1 \boldsymbol{v}_{t-1} + (1 - \beta_1) \boldsymbol{g}_t
   $$

2. **二阶动量（梯度平方的指数加权平均）：**

   $$
   \boldsymbol{s}_t = \beta_2 \boldsymbol{s}_{t-1} + (1 - \beta_2) \boldsymbol{g}_t^2
   $$

3. **偏差修正（因为初始为0，需修正）：**

   $$
   \hat{\boldsymbol{v}}_t = \frac{\boldsymbol{v}_t}{1 - \beta_1^t}, \quad
   \hat{\boldsymbol{s}}_t = \frac{\boldsymbol{s}_t}{1 - \beta_2^t}
   $$

4. **更新参数：**

   $$
   \boldsymbol{\theta}_t = \boldsymbol{\theta}_{t-1} - \eta \cdot \frac{\hat{\boldsymbol{v}}_t}{\sqrt{\hat{\boldsymbol{s}}_t} + \epsilon}
   $$

---

### 🧠 直观理解：

| 组成部分                     | 作用               |
| ------------------------ | ---------------- |
| \$\boldsymbol{v}\_t\$    | 平滑梯度，方向更稳定（动量法）  |
| \$\boldsymbol{s}\_t\$    | 自适应缩放步长（RMSProp） |
| 修正项 \$\hat{v}, \hat{s}\$ | 解决初始时滑动平均偏小问题    |

---

## 🧪 三、代码实现（from scratch）

```python
def adam(params, states, hyperparams):
    beta1, beta2, eps = 0.9, 0.999, 1e-6
    for i, (p, (v, s)) in enumerate(zip(params, states)):
        with torch.no_grad():
            v[:] = beta1 * v + (1 - beta1) * p.grad
            s[:] = beta2 * s + (1 - beta2) * torch.square(p.grad)
            v_hat = v / (1 - beta1 ** (hyperparams['t'] + 1))
            s_hat = s / (1 - beta2 ** (hyperparams['t'] + 1))
            p[:] -= hyperparams['lr'] * v_hat / (torch.sqrt(s_hat) + eps)
        p.grad.zero_()
    hyperparams['t'] += 1
```

### 初始化状态：

```python
def init_adam_states(feature_dim):
    v = torch.zeros((feature_dim, 1))
    s = torch.zeros((feature_dim, 1))
    return (v, s)
```

---

## 📦 四、PyTorch 简洁实现

```python
trainer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8)
```

* `betas` 是动量参数：`(β₁, β₂)`
* `eps` 是避免除以 0 的小数

---

## 📊 五、Adam 与其它优化器的对比

| 优化器      | 是否有动量 | 是否自适应学习率 | 是否需手动设学习率 | 优点                          |
| -------- | ----- | -------- | --------- | --------------------------- |
| SGD      | ❌     | ❌        | ✅         | 简单、广泛使用                     |
| Momentum | ✅     | ❌        | ✅         | 抑制震荡，加速收敛                   |
| AdaGrad  | ❌     | ✅（历史平方）  | ✅         | 稀疏特征有效                      |
| RMSProp  | ❌     | ✅（滑动平均）  | ✅         | 收敛快、通用                      |
| Adadelta | ❌     | ✅        | ❌         | 不依赖学习率、鲁棒性好                 |
| **Adam** | ✅     | ✅        | ✅（但不敏感）   | **综合Momentum+RMSProp，表现优秀** |

---

## 📝 六、总结要点

* **Adam = 动量法 + RMSProp + 偏差修正**
* 学习率调节能力强、方向稳定、默认参数鲁棒
* **默认参数（β₁=0.9, β₂=0.999）通常无需调整**
* 是目前深度学习中最常用的优化算法之一（例如用于Transformer、CNN等）

---

是否需要我再帮你整理第11.11节 **自定义优化器（实战）** 或是做一个 **第11章整体总结笔记**？我可以帮你把优化算法之间的关系画一张图或整理成表格，方便你复习。


当然可以！以下是 **《动手学深度学习》11.10节 Yogi 优化算法** 的完整学习笔记，已整理为 **Markdown 格式**，你可以直接复制进 Word 或 Jupyter Notebook 中使用。

---

# 📘 Yogi优化算法 - 学习笔记（附代码）

## 📌 一、Yogi 是什么？

**Yogi** 是在 2018 年由 *Zaheer, Reddi, Sachan* 等人提出的，
是对 **Adam优化算法的改进版本**，旨在解决 Adam 在非凸场景下的 **不收敛问题**。

---

## ⚠️ 二、Adam 的问题回顾

Adam 在更新二阶矩（梯度平方滑动平均）时，采用公式：

$$
s_t = \beta_2 s_{t-1} + (1 - \beta_2) g_t^2
$$

这种累加方式会导致：

* \$s\_t\$ **持续增大**，尤其当 \$g\_t^2\$ 稍大于 \$s\_{t-1}\$ 时；
* 在训练后期可能造成 **学习率过小、更新停滞**；
* 在某些非凸问题中，**甚至无法收敛**。

---

## ✅ 三、Yogi 的改进公式

Yogi 对二阶矩的更新方式做了如下改进：

$$
s_t = s_{t-1} + (1 - \beta_2) \cdot \operatorname{sgn}(g_t^2 - s_{t-1}) \cdot g_t^2
$$

### 🧠 解释：

* 如果 \$g\_t^2 > s\_{t-1}\$，则增加 \$g\_t^2\$
* 如果 \$g\_t^2 < s\_{t-1}\$，则减少 \$g\_t^2\$
* 这种机制让 \$s\_t\$ 的更新幅度 **更可控、更稳定**

**区别于 Adam：更新方向考虑当前与历史值的比较，而不是单纯增加**

---

## 🧪 四、代码实现（from scratch）

```python
def yogi(params, states, hyperparams):
    beta1, beta2, eps = 0.9, 0.999, 1e-3
    for p, (v, s) in zip(params, states):
        with torch.no_grad():
            # 一阶动量
            v[:] = beta1 * v + (1 - beta1) * p.grad

            # 二阶动量（Yogi 更新方式）
            s[:] = s + (1 - beta2) * torch.sign(torch.square(p.grad) - s) * torch.square(p.grad)

            # 偏差修正
            v_corr = v / (1 - beta1 ** hyperparams['t'])
            s_corr = s / (1 - beta2 ** hyperparams['t'])

            # 参数更新
            p[:] -= hyperparams['lr'] * v_corr / (torch.sqrt(s_corr) + eps)

        p.grad.zero_()
    hyperparams['t'] += 1
```

### 初始化状态：

```python
def init_adam_states(feature_dim):
    v = torch.zeros((feature_dim, 1))
    s = torch.zeros((feature_dim, 1))
    return (v, s)
```

---

## 🚀 五、使用示例

```python
data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)

d2l.train_ch11(
    yogi,
    init_adam_states(feature_dim),
    {'lr': 0.01, 't': 1},
    data_iter,
    feature_dim
)
```

输出示例：

```
loss: 0.243, 0.022 sec/epoch
```

并伴有 loss 曲线图收敛效果可视化。

---

## 📊 六、与 Adam 的对比总结

| 特性     | Adam    | Yogi             |
| ------ | ------- | ---------------- |
| 一阶动量   | ✅       | ✅                |
| 二阶动量更新 | 简单指数加权  | 使用 sign 控制增减，更稳定 |
| 收敛性    | 有时发散或停滞 | 更鲁棒，能避免学习率过小     |
| 应用场景   | 通用      | 非凸优化、训练后期收敛不佳的问题 |

---

## 📝 七、总结

* **Yogi 是一种更稳定的自适应优化器**
* 相较于 Adam，它更好地控制了二阶矩的增长速度
* 避免了收敛过早、震荡等问题
* 被广泛用于大型模型训练（如 NLP 中的 BERT）

---

是否需要我将 Adam、Yogi、RMSProp、AdaGrad 做一张优化器对比图表，或整理成一页“优化算法总览笔记”？我可以继续帮助你总结第11章的核心要点。
